In [ ]:
import pandas as pd

# 指定文件路径
file_path = r'data/raw/shareholders\HLD_Shareholders.csv'

# 使用 pandas 读取 CSV 文件
df = pd.read_csv(file_path,error_bad_lines=False)

# 显示数据的前几行，确认数据已成功读取
print(df.head())

# 首先是筛选日期

In [ ]:
# Convert the 'ReptDt' column to datetime format
df['Reptdt'] = pd.to_datetime(df['Reptdt'])

# Filter the DataFrame for the date range from '2015-12-31' to '2016-12-31'
filtered_df = df[(df['Reptdt'] >= '2015-12-31') & (df['Reptdt'] <= '2016-12-31')]

filtered_df

# 进一步筛选股票163

In [ ]:
# 定义股票列表
stock_list =[601088, 2049, 2050, 2, 513, 519, 601607, 600585, 601100, 526, 600079, 
             605198, 603667, 21, 600600, 2074, 601628, 2594, 34, 2597, 550, 2603, 
             2605, 600111, 2612, 2555, 568, 603706, 63, 2624, 600129, 2625, 600132, 
             603721, 600141, 603728, 600660, 2646, 603738, 603739, 2653, 2657, 2151, 
             605288, 605289, 2153, 603755, 2664, 600685, 2158, 600690, 605298, 2675, 
             600183, 600696, 605305, 600699, 636, 2683, 600188, 603776, 600199, 651, 
             601231, 2192, 2705, 661, 2709, 603799, 605336, 2714, 605339, 2715, 603813, 
             2213, 2727, 600745, 2223, 2735, 1201, 2737, 603825, 1205, 2230, 1206, 2229, 
             603833, 1209, 2232, 2236, 2240, 2241, 600771, 605388, 601808, 605398, 2262, 
             603868, 2271, 600801, 2275, 601318, 603369, 603882, 603883, 2281, 2287, 756, 
             603383, 601336, 603895, 762, 603899, 600316, 603392, 768, 776, 601865, 600332, 
             1296, 600338, 601877, 603931, 799, 600362, 600875, 2864, 605488, 818, 2867, 
             2868, 603960, 2873, 600895, 605507, 2884, 2886, 2890, 1872, 2902, 603991, 603486, 
             603489, 2913, 2916, 2918, 2409, 2921, 2930, 2932, 603508, 600438, 887, 2938, 
             1914, 603517, 2942, 2943, 2432, 2949, 2952, 603530, 2955, 605068, 2444, 603533, 
             2959, 2960, 400, 913, 403, 2957, 2961, 408, 921, 2970, 2971, 2460, 600988, 2463, 
             2466, 2979, 422, 423, 2983, 2985, 2986, 2475, 2987, 605099, 935, 2991, 2484, 
             2997, 2487, 3001, 605116, 601021, 605118, 603583, 3009, 605122, 603587, 603589, 
             3013, 600007, 3019, 2508, 3022, 2001, 2518, 2007, 600536, 2008, 2009, 603611, 
             3032, 603613, 600031, 2531, 600036, 2534, 2025, 601066, 2030, 600559, 2032, 
             2544, 601595, 600060]

# 筛选出Stkcd在stock_list中的行
filtered_stkcd_df = filtered_df[filtered_df['Stkcd'].isin(stock_list)]

In [ ]:
filtered_stkcd_df

In [ ]:
filtered_stkcd_df[filtered_stkcd_df['Stkcd']==1]

# 对于这个表格，每只股票的Reptdt均有很多个时间，每个时间都有很多个股东S0301a（可能存在重复） ，现在筛选出每只股票在所有时间都存在的股东，提取出这些行（只提取出第一次出现的那一行）

In [ ]:
df=filtered_stkcd_df

In [ ]:
import pandas as pd

# 假设df是你的数据框
# 我们首先对每只股票(Stkcd)进行分组，并对股东(S0301a)在所有日期中的出现情况进行筛选

# 对每只股票(Stkcd)进行分组，找到每只股票所有时间都有的股东，并只保留第一次出现的行
def find_common_shareholders_first_occurrence(df):
    result = []
    
    # 对每个股票进行分组
    for stkcd, group in df.groupby('Stkcd'):
        # 找到所有时间都出现的股东
        common_shareholders = group.groupby('S0301a').filter(lambda x: x['Reptdt'].nunique() == group['Reptdt'].nunique())
        
        # 只保留第一次出现的股东
        first_occurrence = common_shareholders.drop_duplicates(subset='S0301a', keep='first')
        
        result.append(first_occurrence)
    
    # 将所有股票的结果合并
    return pd.concat(result)

# 使用函数筛选出每只股票所有时间都有的股东，并保留第一次出现的行
filtered_df = find_common_shareholders_first_occurrence(df)

# 输出结果
filtered_df


In [ ]:
filtered_df[filtered_df['Stkcd']==2]

In [ ]:
import pandas as pd

# 假设filtered_df是你的原始数据框
# 查找重复股东
shareholder_group = filtered_df.groupby('S0301a')['Stkcd'].nunique()

# 过滤出出现在多个股票中的股东
duplicate_shareholders = shareholder_group[shareholder_group > 1].index

# 提取这些重复股东出现在的数据行，并配对股票序号
duplicate_df = filtered_df[filtered_df['S0301a'].isin(duplicate_shareholders)]

# 构建结果，包含股票对和股东名
result = []

for shareholder in duplicate_shareholders:
    stocks = duplicate_df[duplicate_df['S0301a'] == shareholder]['Stkcd'].unique()
    if len(stocks) > 1:
        for i in range(len(stocks)):
            for j in range(i + 1, len(stocks)):
                result.append((stocks[i], stocks[j], shareholder))

# 将结果转换为数据框
duplicate_stocks_df = pd.DataFrame(result, columns=['Stock1', 'Stock2', 'Shareholder'])

# 展示结果
duplicate_stocks_df

# 看看总股东有多少（21个）

In [ ]:
len(duplicate_stocks_df['Shareholder'].unique())

In [ ]:
duplicate_stocks_df['Shareholder'].unique()

# 看看股票A-股东-股票B边的数量1937

In [ ]:
import pandas as pd

# 假设duplicate_stocks_df是你之前生成的数据框

# 创建一个无序对：每一行中Stock1和Stock2按大小排序，形成元组
duplicate_stocks_df['Pair'] = duplicate_stocks_df.apply(lambda row: tuple(sorted([row['Stock1'], row['Stock2']])), axis=1)

# 去除重复的元组对
unique_pairs_df = duplicate_stocks_df.drop_duplicates(subset='Pair')

# 删除临时生成的'Pair'列
unique_pairs_df = unique_pairs_df.drop(columns=['Pair'])

# 展示结果
unique_pairs_df

# 现在我要构造矩阵，注意不能直接用filtered_df里面的股东，因为我现在只要同时投资两只股票以上的股东，如果用前者，则会筛选出很多只投资了一个股票的股东

In [ ]:
import pandas as pd
import numpy as np

# 假设 df1 是你给出的 DataFrame
# 从 df1 中提取唯一的名字
unique_names = duplicate_stocks_df['Shareholder'].unique()
# 创建一个大小为 (len(stock_list), len(unique_names)) 的矩阵，初始值全为 0
matrix = pd.DataFrame(0, index=stock_list, columns=unique_names)

# 遍历 df1，将 Stkcd 和 Name 组合的位置填充为 1
for _, row in filtered_df.iterrows():
    stkcd = row['Stkcd']
    name = row['S0301a']
    if stkcd in stock_list:
        matrix.at[stkcd, name] = 1

# 输出所有列名
columns_list = matrix.columns.tolist()

# 返回结果
matrix

In [ ]:
# 取出所有行，并只选择前31列
matrix_first_31_columns = matrix.iloc[:, :21]

# 展示结果
matrix_first_31_columns

In [ ]:
unique_names

In [ ]:
# 创建两个列表用于存储行索引和列索引
row_indices = []
column_indices = []

# 遍历矩阵，检查每个元素是否为 1，如果是 1 则记录其行索引和列索引
for row_index, row in matrix_first_31_columns.iterrows():
    for col_index, value in row.items():
        if value == 1:
            row_indices.append(row_index)
            column_indices.append(col_index)

# 将行索引和列索引组合成一个大列表
result = [row_indices, column_indices]

# 输出结果
print(result)

In [ ]:
# 创建两个列表用于存储行和列的序号
row_indices = []
column_indices = []

# 遍历矩阵，检查每个元素是否为 1，如果是 1 则记录其行和列的序号
for row_idx, (row_name, row) in enumerate(matrix_first_31_columns.iterrows()):  # 使用 enumerate 获取行的序号
    for col_idx, (col_name, value) in enumerate(row.items()):  # 使用 enumerate 获取列的序号
        if value == 1:
            row_indices.append(row_idx )  # 行序号
            column_indices.append(col_idx+163 )  # 列序号

# 将行序号和列序号组合成一个大列表
result = [row_indices, column_indices]

# 输出结果
result

# 现在构建第二张表格

In [ ]:
filtered_df

对于这个表格，stock_list 作为行名，names作为列名，构造一个全0的矩阵，矩阵填法具体为：例如第一行，对应的Stkcd为1，Name为冯杰，则在这个矩阵对应位置填1（整数类型），

In [ ]:
stock_list

In [ ]:
unique_names

In [ ]:
# 创建全0的矩阵，行名为stock_list，列名为names
matrix3 = pd.DataFrame(np.zeros((len(stock_list), len(unique_names)), dtype=int), index=stock_list, columns=unique_names)

# 遍历result_filtered，根据Stkcd和Name填充矩阵
for _, row in filtered_df.iterrows():
    stkcd = row['Stkcd']
    name = row['S0301a']
    
    if stkcd in stock_list and name in unique_names:
        matrix3.at[stkcd, name] = 1

# 返回矩阵中的1的数量
count_of_ones = matrix3.values.sum()
count_of_ones

In [ ]:
matrix3